# 👁️ AI Smart Camera v0.2.1: Reliability, Security, Performance & Reproducibility

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tahadeab-ai/ai-smart-camera/blob/main/notebooks/01_ai_smart_camera_colab.ipynb)
[![GitHub License](https://img.shields.io/badge/License-MIT-blue.svg)](https://github.com/tahadeab-ai/ai-smart-camera/blob/main/LICENSE)
[![FastAPI](https://img.shields.io/badge/Backend-FastAPI-009688.svg)](https://fastapi.tiangolo.com/)
[![React 18](https://img.shields.io/badge/Frontend-React%2018%20%2B%20TypeScript-61dafb.svg)](https://react.dev/)
[![YOLO](https://img.shields.io/badge/Model-YOLOv8%20%7C%20YOLO11-orange.svg)](https://docs.ultralytics.com/)
[![Supervision](https://img.shields.io/badge/Vision-Supervision-green.svg)](https://supervision.roboflow.com/)

Welcome to the official **Google Colab Interactive Execution Environment** for **AI Smart Camera v0.2.1**.

### v0.2.1 Hardening Highlights
- 🔒 **Hardened Image Ingestion**: Early Content-Length check, 64KB bounded chunk streaming (20MB max), magic byte validation, and 4096x4096px bounds preventing decompression bombs.
- 🎫 **WebSocket Single-Use Tickets**: Master access token never exposed in WebSocket URLs or browser history; tickets issued via `POST /api/v1/auth/ws-ticket` (30s TTL, single-use).
- 🛡️ **WebSocket Origin Validation**: Handshake verification against permitted CORS origins prevents Cross-Site WebSocket Hijacking.
- ⚡ **Non-Blocking Threadpool Vision**: CPU/GPU-bound image decoding, YOLO inference, and annotation offloaded from asyncio loop via `run_in_threadpool`, keeping health checks responsive under heavy inference.
- ⏱️ **Atomic Concurrency Controls**: Lock-guarded slot reservations in `StreamManager` preventing race conditions upon concurrent WebSocket handshakes.
- 🎛️ **Rate Limiting**: Sliding window token bucket limiter (10 req/min) on `/api/v1/detect/image` and `/api/v1/auth/ws-ticket`.
- 📦 **Deterministic Builds**: Pinned python packages in `requirements.txt` and verified `frontend/package-lock.json` for `npm ci`.

---

## 1. Environment & Hardware Inspection
Inspect the Colab runtime environment, including CPU, RAM, and CUDA GPU acceleration (with automatic CPU fallback).

In [ ]:
import sys
import os
import platform
import psutil
import torch

print('=' * 60)
print('AI SMART CAMERA v0.2.1 - HARDWARE & ENVIRONMENT AUDIT')
print('=' * 60)
print(f'Python Version   : {platform.python_version()} ({platform.python_implementation()})')
print(f'Operating System : {platform.system()} {platform.release()}')
print(f'CPU Logical Cores: {psutil.cpu_count(logical=True)}')
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f'System Memory    : {ram_gb:.2f} GB RAM')

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    device_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f'CUDA Accelerator : {device_name} ({device_mem:.2f} GB VRAM)')
    DEFAULT_DEVICE = 'cuda'
else:
    print('CUDA Accelerator : Not detected. Using CPU execution mode.')
    DEFAULT_DEVICE = 'cpu'
print('=' * 60)


## 2. Clone Repository
Clone the permanent GitHub source of truth (`tahadeab-ai/ai-smart-camera`).

In [ ]:
%cd /content
!rm -rf ai-smart-camera
!git clone https://github.com/tahadeab-ai/ai-smart-camera.git
%cd ai-smart-camera
!git status


## 3. Install Pinned Python Dependencies
Install verified Python packages from `requirements-dev.txt` for development, testing, and benchmarking.

In [ ]:
%cd /content/ai-smart-camera
!pip install --upgrade pip
!pip install -r requirements-dev.txt


## 4. Model Checksum & Manifest Verification
Download and verify YOLO model weights against the SHA-256 cryptographic hashes specified in `models/manifest.yaml`.

In [ ]:
%cd /content/ai-smart-camera
from app.core.model_loader import ModelLoader

loader = ModelLoader()
print('Verifying models against manifest checksums:')
for model_key in ['yolov8n', 'yolo11n']:
    meta = loader.get_model_metadata(model_key)
    path = loader.ensure_model_available(model_key)
    sha = loader._calculate_sha256(path)
    assert sha == meta['sha256'], f'Checksum mismatch for {model_key}!'
    print(f'  [PASS] {model_key}: {path.name} (SHA-256: {sha[:16]}...)')


## 5. Automated Python Test Suite Execution
Run the complete unit and integration test suite with test coverage reporting.

In [ ]:
%cd /content/ai-smart-camera
!pytest -v tests/ --cov=app --cov-report=term-missing


## 6. Compile React 18 + TypeScript Frontend (`npm ci`)
Install deterministic npm dependencies and compile the production single-page application into `frontend/dist`.

In [ ]:
%cd /content/ai-smart-camera/frontend
!npm ci
!npm test
!npm run build
!ls -la /content/ai-smart-camera/frontend/dist


## 7. Run Reproducible Hardware Benchmarks
Execute the automated benchmarking framework measuring P50/P95 latency, statistical variance, and throughput FPS on this Colab runtime.

In [ ]:
%cd /content/ai-smart-camera
!python benchmarks/benchmark.py --model yolov8n --device {DEFAULT_DEVICE} --runs 30
!cat benchmarks/benchmark_report.md


## 8. Generate Random Deployment Access Token
Generate a cryptographically secure random session token for protecting the public tunnel endpoint.

In [ ]:
import secrets

DEPLOY_TOKEN = secrets.token_urlsafe(16)
os.environ['ACCESS_TOKEN'] = DEPLOY_TOKEN
print('=' * 60)
print(f'DEPLOYMENT ACCESS TOKEN: {DEPLOY_TOKEN}')
print('Use this token in the React UI Settings modal to authorize access.')
print('=' * 60)


## 9. Launch FastAPI Backend in Background
Start the FastAPI server on port 7860 serving both the API endpoints and the compiled React frontend.

In [ ]:
%cd /content/ai-smart-camera
import subprocess
import time
import urllib.request

# Terminate any existing server processes
!pkill -f 'app.main' || true

server_proc = subprocess.Popen(
    [
        sys.executable, '-m', 'app.main',
        '--host', '0.0.0.0',
        '--port', '7860',
        '--model', 'yolov8n',
        '--device', DEFAULT_DEVICE,
    ],
    stdout=open('/content/server.log', 'w'),
    stderr=subprocess.STDOUT,
)

print('Waiting for FastAPI server to initialize...')
for _ in range(25):
    try:
        with urllib.request.urlopen('http://localhost:7860/api/v1/health/live') as res:
            if res.status == 200:
                print('FastAPI server is running and healthy on http://localhost:7860')
                break
    except Exception:
        time.sleep(1)
else:
    print('Warning: Server startup timed out. Check /content/server.log')


## 10. Expose via Cloudflare Quick Tunnel
Create a secure public HTTPS URL using Cloudflare Quick Tunnel to access the full application from your browser or mobile camera.

In [ ]:
import re
import subprocess
import time

!which cloudflared || (wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb)

!pkill -f 'cloudflared' || true
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:7860'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

print('=' * 65)
print(f'PUBLIC HTTPS APPLICATION URL : {public_url}')
print(f'DEPLOYMENT ACCESS TOKEN      : {DEPLOY_TOKEN}')
print('=' * 65)


## 11. Clean Server Shutdown
Safely terminate all background processes and clean up runtime resources when finished.

In [ ]:
!pkill -f 'cloudflared' || true
!pkill -f 'app.main' || true
print('All background processes stopped. Runtime clean.')
